# Colab Ollama Bridge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vidoxlabs/colab-ollama-bridge/blob/main/notebooks/colab_ollama.ipynb)

Turns an interactive Google Colab GPU runtime into an authenticated, OpenAI-compatible Ollama endpoint protected by Cloudflare Tunnel.

> [!IMPORTANT]
> **Security Invariants**:
> - Ollama binds only to loopback (`127.0.0.1:11434`). It is never exposed directly to the internet.
> - All public or tunneled traffic must pass through the loopback authentication proxy (`127.0.0.1:11435`) which enforces `BRIDGE_API_KEY`.
> - Quick Tunnels are ephemeral and strictly for development/testing.

> [!WARNING]
> **Colab Lifecycle Notice**:
> Google Colab is an interactive, ephemeral environment. This notebook does not keep Colab alive, simulate activity, or prevent idle/maximum lifetime termination.

In [ ]:
# Cell 1: Verify Google Colab runtime and NVIDIA GPU
import sys, subprocess

if 'google.colab' not in sys.modules:
    print("[NOTICE] Not running in standard Colab environment (running locally or self-hosted).")

try:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
    print("NVIDIA GPU detected successfully:")
    print(smi.stdout.splitlines()[0])
    print(smi.stdout.splitlines()[8] if len(smi.stdout.splitlines()) > 8 else "")
except Exception as err:
    sys.exit("[ERROR] No NVIDIA GPU detected. In Colab, select: Runtime > Change runtime type > T4/L4/A100 GPU.")

In [ ]:
# Cell 2: Fetch and verify bootstrap assets (or locate local clone)
import os, urllib.request, hashlib
from pathlib import Path

REPO_ROOT = Path("/content/colab-ollama-bridge")
if not REPO_ROOT.exists():
    # Check if in repo workspace
    if Path("scripts/bootstrap.sh").exists():
        REPO_ROOT = Path(".").resolve()
    else:
        print("Cloning repository...")
        subprocess.run(["git", "clone", "https://github.com/Vidoxlabs/colab-ollama-bridge.git", str(REPO_ROOT)], check=True)

bootstrap_script = REPO_ROOT / "scripts" / "bootstrap.sh"
if not bootstrap_script.exists():
    sys.exit(f"[ERROR] bootstrap.sh not found at {bootstrap_script}")
print(f"Bootstrap script ready at: {bootstrap_script}")

In [ ]:
# Cell 3: Securely resolve credentials (never print secrets)
import getpass

bridge_api_key = None
tunnel_token = None

try:
    from google.colab import userdata
    try:
        bridge_api_key = userdata.get('BRIDGE_API_KEY')
    except Exception:
        pass
    try:
        tunnel_token = userdata.get('TUNNEL_TOKEN')
    except Exception:
        pass
except ImportError:
    pass

if not bridge_api_key:
    bridge_api_key = os.environ.get('BRIDGE_API_KEY')

if not bridge_api_key:
    bridge_api_key = getpass.getpass("Enter BRIDGE_API_KEY (minimum 16 characters): ")

if not bridge_api_key or len(bridge_api_key.strip()) < 16:
    sys.exit("[ERROR] BRIDGE_API_KEY is required and must be at least 16 characters.")

os.environ['BRIDGE_API_KEY'] = bridge_api_key.strip()
if tunnel_token:
    os.environ['TUNNEL_TOKEN'] = tunnel_token.strip()
print("Credentials securely configured in memory.")

In [ ]:
# Cell 4: Invoke shared bootstrap in foreground orchestrator mode
import os, subprocess

env = os.environ.copy()
env['SKIP_SUPERVISE'] = 'true'
bootstrap_cmd = ["bash", str(REPO_ROOT / "scripts" / "bootstrap.sh")]

result = subprocess.run(bootstrap_cmd, env=env, cwd=str(REPO_ROOT))
if result.returncode != 0:
    sys.exit(f"[ERROR] Bootstrap exited with code {result.returncode}")

In [ ]:
# Cell 5: Display connection health and OpenCode snippet (environment references only)
from pathlib import Path

state_dir = Path(os.environ.get('BRIDGE_STATE_DIR', '/tmp/bridge_state'))
log_path = state_dir / "cloudflared.log"
tunnel_url = "Unknown"

if log_path.exists():
    import re
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_path.read_text())
    if matches:
        tunnel_url = matches[0]

print("=" * 70)
print(f"Colab Ollama Bridge Active: {tunnel_url}")
print("=" * 70)
print("Export in your local terminal:")
print(f'  export COLAB_OLLAMA_BASE_URL="{tunnel_url}"')
print('  export COLAB_BRIDGE_API_KEY="<your-bridge-key>"')
print("=" * 70)

In [ ]:
# Cell 6: Authenticated inference smoke test
import json, urllib.request

bridge_bind = os.environ.get('BRIDGE_BIND', '127.0.0.1:11435')
key_path = Path(os.environ.get('BRIDGE_STATE_DIR', '/tmp/bridge_state')) / "bridge_api.key"
api_key = key_path.read_text().strip()

req = urllib.request.Request(
    f"http://{bridge_bind}/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)
with urllib.request.urlopen(req) as resp:
    data = json.loads(resp.read().decode('utf-8'))
    models = [m.get('id') for m in data.get('data', [])]
    print(f"Smoke test succeeded! Available models: {models}")

In [ ]:
# Cell 7: Monitor owned process health (Interrupt cell to stop)
# Note: This cell reports process health. It does NOT prevent Colab session termination.
import time, sys
sys.path.insert(0, str(REPO_ROOT))
from src.supervisor import ProcessSupervisor

supervisor = ProcessSupervisor(state_dir=str(state_dir))
print("Monitoring bridge processes. Press Stop/Interrupt to pause.")
try:
    while True:
        healthy = supervisor.check_health()
        status = "HEALTHY" if healthy else "DEGRADED"
        print(f"[STATUS] Bridge health: {status}")
        time.sleep(15)
except KeyboardInterrupt:
    print("\nMonitor interrupted by user.")

In [ ]:
# Cell 8: Graceful shutdown and secret purge
supervisor.stop_all()
print("All bridge processes stopped and temporary secret files purged.")